# 12 Registro rigido por mascaras - todos los sujetos

Este notebook aplica el baseline rigido H&E -> HSI a todos los sujetos disponibles en `Imagenes_a_escala`.

La HSI es la imagen fija y la H&E es la imagen movil. La escala no se optimiza porque ya fue igualada en el preprocesado.

In [ ]:
from pathlib import Path
import csv
import json

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

plt.rcParams['figure.dpi'] = 120

BASE_DIR = Path(r'Registration\DeepLearning')
INPUT_DIR = BASE_DIR / 'Imagenes_a_escala'
OUTPUT_ROOT = BASE_DIR / 'Tecnicas_registration' / '00_baseline_clasico' / 'outputs' / 'outputs_registro_rigido_mascaras'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SUBJECTS = ['SB012', 'SB013', 'SB017', 'SB018', 'SB019', 'SB020']

print('Entrada:', INPUT_DIR)
print('Salida:', OUTPUT_ROOT)

## Funciones auxiliares

In [ ]:
def load_rgb(path):
    return np.asarray(Image.open(path).convert('RGB'), dtype=np.uint8)


def load_mask(path):
    return np.asarray(Image.open(path).convert('L')) > 0


def centroid_xy(mask):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        raise ValueError('Mascara vacia')
    return np.array([xs.mean(), ys.mean()], dtype=np.float64)


def rigid_matrix_moving_to_fixed(moving_mask, fixed_mask, angle_deg):
    moving_centroid = centroid_xy(moving_mask)
    fixed_centroid = centroid_xy(fixed_mask)
    matrix = cv2.getRotationMatrix2D(tuple(moving_centroid), float(angle_deg), 1.0)
    matrix[:, 2] += fixed_centroid - moving_centroid
    return matrix


def warp_to_fixed_canvas(image, fixed_shape, matrix, is_mask=False):
    interpolation = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    if is_mask:
        src = image.astype(np.uint8) * 255
        warped = cv2.warpAffine(
            src,
            matrix,
            dsize=(fixed_shape[1], fixed_shape[0]),
            flags=interpolation,
            borderMode=cv2.BORDER_CONSTANT,
            borderValue=0,
        )
        return warped > 0

    return cv2.warpAffine(
        image,
        matrix,
        dsize=(fixed_shape[1], fixed_shape[0]),
        flags=interpolation,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0),
    )


def dice_score(mask_a, mask_b):
    a = np.asarray(mask_a, dtype=bool)
    b = np.asarray(mask_b, dtype=bool)
    denom = int(a.sum()) + int(b.sum())
    if denom == 0:
        return 1.0
    return float(2 * np.logical_and(a, b).sum() / denom)


def iou_score(mask_a, mask_b):
    a = np.asarray(mask_a, dtype=bool)
    b = np.asarray(mask_b, dtype=bool)
    union = np.logical_or(a, b).sum()
    if union == 0:
        return 1.0
    return float(np.logical_and(a, b).sum() / union)


def contour_mask(mask, thickness=1):
    mask_u8 = np.asarray(mask, dtype=np.uint8) * 255
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    out = np.zeros_like(mask_u8)
    cv2.drawContours(out, contours, contourIdx=-1, color=255, thickness=int(thickness))
    return out > 0


def symmetric_contour_distance(mask_a, mask_b):
    ca = contour_mask(mask_a)
    cb = contour_mask(mask_b)
    if ca.sum() == 0 or cb.sum() == 0:
        return {'mean_px': np.nan, 'p95_px': np.nan}

    dist_to_b = cv2.distanceTransform((~cb).astype(np.uint8), cv2.DIST_L2, 5)
    dist_to_a = cv2.distanceTransform((~ca).astype(np.uint8), cv2.DIST_L2, 5)
    distances = np.concatenate([dist_to_b[ca], dist_to_a[cb]])
    return {
        'mean_px': float(np.mean(distances)),
        'p95_px': float(np.percentile(distances, 95)),
    }


def save_mask(path, mask):
    Image.fromarray((np.asarray(mask) > 0).astype(np.uint8) * 255).save(path)


def overlay_rgb_images(fixed_rgb, moving_rgb, fixed_mask=None, moving_mask=None):
    fixed_gray = cv2.cvtColor(fixed_rgb, cv2.COLOR_RGB2GRAY)
    moving_gray = cv2.cvtColor(moving_rgb, cv2.COLOR_RGB2GRAY)
    fixed_norm = cv2.normalize(fixed_gray, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    moving_norm = cv2.normalize(moving_gray, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    overlay = np.zeros((*fixed_gray.shape, 3), dtype=np.uint8)
    overlay[..., 0] = fixed_norm
    overlay[..., 1] = moving_norm
    overlay[..., 2] = fixed_norm // 3

    if fixed_mask is not None:
        overlay[contour_mask(fixed_mask, thickness=2)] = np.array([255, 0, 0], dtype=np.uint8)
    if moving_mask is not None:
        overlay[contour_mask(moving_mask, thickness=2)] = np.array([0, 255, 255], dtype=np.uint8)
    return overlay


def contours_overlay_image(fixed_mask, moving_mask):
    out = np.zeros((*fixed_mask.shape, 3), dtype=np.uint8)
    out[contour_mask(fixed_mask, thickness=2)] = np.array([255, 0, 0], dtype=np.uint8)
    out[contour_mask(moving_mask, thickness=2)] = np.array([0, 255, 255], dtype=np.uint8)
    return out


def write_csv(path, rows, fieldnames):
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)

## Procesar un sujeto

In [ ]:
def subject_paths(subject_id):
    subject_dir = INPUT_DIR / subject_id
    return {
        'subject_dir': subject_dir,
        'he_rgb': subject_dir / f'{subject_id}_h&e.png',
        'hsi_rgb': subject_dir / f'{subject_id}_hsi.png',
        'he_mask': subject_dir / f'{subject_id}_he_mask.png',
        'hsi_mask': subject_dir / f'{subject_id}_hsi_mask.png',
        'metadata': subject_dir / f'{subject_id}_metadata.json',
    }


def evaluate_angle(he_mask, hsi_mask, angle_deg):
    matrix = rigid_matrix_moving_to_fixed(he_mask, hsi_mask, angle_deg)
    warped_mask = warp_to_fixed_canvas(he_mask, hsi_mask.shape, matrix, is_mask=True)
    return {
        'angle_deg': float(angle_deg),
        'dice': dice_score(warped_mask, hsi_mask),
        'iou': iou_score(warped_mask, hsi_mask),
    }


def search_best_angle(he_mask, hsi_mask, coarse_step=2.0, fine_radius=3.0, fine_step=0.25):
    coarse_angles = np.arange(-180.0, 180.0 + 1e-6, coarse_step)
    coarse_results = [evaluate_angle(he_mask, hsi_mask, angle) for angle in coarse_angles]
    best_coarse = max(coarse_results, key=lambda row: row['dice'])

    fine_angles = np.arange(
        best_coarse['angle_deg'] - fine_radius,
        best_coarse['angle_deg'] + fine_radius + 1e-6,
        fine_step,
    )
    fine_results = [evaluate_angle(he_mask, hsi_mask, angle) for angle in fine_angles]
    best_fine = max(fine_results, key=lambda row: row['dice'])
    return best_fine, coarse_results + fine_results


def process_subject(subject_id):
    paths = subject_paths(subject_id)
    out_dir = OUTPUT_ROOT / subject_id
    out_dir.mkdir(parents=True, exist_ok=True)

    he_rgb = load_rgb(paths['he_rgb'])
    hsi_rgb = load_rgb(paths['hsi_rgb'])
    he_mask = load_mask(paths['he_mask'])
    hsi_mask = load_mask(paths['hsi_mask'])
    metadata = json.loads(paths['metadata'].read_text(encoding='utf-8'))

    best, angle_results = search_best_angle(he_mask, hsi_mask)
    best_angle = best['angle_deg']
    best_matrix = rigid_matrix_moving_to_fixed(he_mask, hsi_mask, best_angle)
    initial_matrix = rigid_matrix_moving_to_fixed(he_mask, hsi_mask, 0.0)

    he_mask_initial = warp_to_fixed_canvas(he_mask, hsi_mask.shape, initial_matrix, is_mask=True)
    he_mask_rigid = warp_to_fixed_canvas(he_mask, hsi_mask.shape, best_matrix, is_mask=True)
    he_rgb_rigid = warp_to_fixed_canvas(he_rgb, hsi_mask.shape, best_matrix, is_mask=False)

    contour_distance = symmetric_contour_distance(he_mask_rigid, hsi_mask)
    overlay = overlay_rgb_images(hsi_rgb, he_rgb_rigid, hsi_mask, he_mask_rigid)
    contour_overlay = contours_overlay_image(hsi_mask, he_mask_rigid)

    files = {
        'he_rgb_rigid': out_dir / f'{subject_id}_he_rigid_to_hsi.png',
        'he_mask_rigid': out_dir / f'{subject_id}_he_mask_rigid_to_hsi.png',
        'overlay_rgb': out_dir / f'{subject_id}_overlay_rgb_rigid.png',
        'overlay_contours': out_dir / f'{subject_id}_overlay_contours_rigid.png',
        'metrics': out_dir / f'{subject_id}_rigid_metrics.json',
        'angle_search': out_dir / f'{subject_id}_angle_search.csv',
    }

    Image.fromarray(he_rgb_rigid).save(files['he_rgb_rigid'])
    save_mask(files['he_mask_rigid'], he_mask_rigid)
    Image.fromarray(overlay).save(files['overlay_rgb'])
    Image.fromarray(contour_overlay).save(files['overlay_contours'])
    write_csv(files['angle_search'], angle_results, ['angle_deg', 'dice', 'iou'])

    metrics = {
        'subject_id': subject_id,
        'registration_direction': 'he_to_hsi',
        'fixed_image': 'hsi',
        'moving_image': 'he',
        'best_angle_deg': float(best_angle),
        'transform_matrix_2x3': best_matrix.tolist(),
        'dice_initial_angle_0': dice_score(he_mask_initial, hsi_mask),
        'iou_initial_angle_0': iou_score(he_mask_initial, hsi_mask),
        'dice_rigid': dice_score(he_mask_rigid, hsi_mask),
        'iou_rigid': iou_score(he_mask_rigid, hsi_mask),
        'contour_mean_distance_px': contour_distance['mean_px'],
        'contour_p95_distance_px': contour_distance['p95_px'],
        'he_rgb_shape': list(he_rgb.shape),
        'hsi_rgb_shape': list(hsi_rgb.shape),
        'target_px_per_cm': float(metadata['target_px_per_cm']),
        'output_dir': str(out_dir),
        'overlay_rgb_path': str(files['overlay_rgb']),
        'overlay_contours_path': str(files['overlay_contours']),
    }
    files['metrics'].write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    return metrics, overlay, contour_overlay

## Ejecutar todos los sujetos

In [ ]:
all_metrics = []
overlay_images = {}
contour_images = {}

for subject_id in SUBJECTS:
    print(f'=== {subject_id} ===')
    metrics, overlay, contour_overlay = process_subject(subject_id)
    all_metrics.append(metrics)
    overlay_images[subject_id] = overlay
    contour_images[subject_id] = contour_overlay
    print(
        f"angle={metrics['best_angle_deg']:.2f} | "
        f"Dice {metrics['dice_initial_angle_0']:.4f} -> {metrics['dice_rigid']:.4f} | "
        f"IoU {metrics['iou_initial_angle_0']:.4f} -> {metrics['iou_rigid']:.4f} | "
        f"contour_mean={metrics['contour_mean_distance_px']:.2f}px"
    )

summary_csv = OUTPUT_ROOT / 'rigid_mask_summary.csv'
summary_json = OUTPUT_ROOT / 'rigid_mask_summary.json'
summary_fields = [
    'subject_id',
    'best_angle_deg',
    'dice_initial_angle_0',
    'dice_rigid',
    'iou_initial_angle_0',
    'iou_rigid',
    'contour_mean_distance_px',
    'contour_p95_distance_px',
    'target_px_per_cm',
    'overlay_rgb_path',
    'overlay_contours_path',
]
write_csv(summary_csv, all_metrics, summary_fields)
summary_json.write_text(json.dumps(all_metrics, indent=2), encoding='utf-8')

print('Resumen CSV:', summary_csv)
print('Resumen JSON:', summary_json)

## Tabla resumen

In [ ]:
for row in all_metrics:
    print(
        f"{row['subject_id']}: angle={row['best_angle_deg']:7.2f} deg | "
        f"Dice={row['dice_rigid']:.4f} | IoU={row['iou_rigid']:.4f} | "
        f"contour_mean={row['contour_mean_distance_px']:.2f}px | "
        f"contour_p95={row['contour_p95_distance_px']:.2f}px"
    )

In [ ]:
subject_ids = [row['subject_id'] for row in all_metrics]
dice_initial = [row['dice_initial_angle_0'] for row in all_metrics]
dice_rigid = [row['dice_rigid'] for row in all_metrics]
angles = [row['best_angle_deg'] for row in all_metrics]

x = np.arange(len(subject_ids))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
axes[0].bar(x - width / 2, dice_initial, width, label='centroides sin rotacion')
axes[0].bar(x + width / 2, dice_rigid, width, label='rigido final')
axes[0].set_xticks(x, subject_ids, rotation=30)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Dice')
axes[0].set_title('Solapamiento de mascaras')
axes[0].legend(fontsize=8)
axes[0].grid(axis='y', alpha=0.25)

axes[1].bar(x, angles)
axes[1].set_xticks(x, subject_ids, rotation=30)
axes[1].set_ylabel('grados')
axes[1].set_title('Angulo estimado H&E -> HSI')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].grid(axis='y', alpha=0.25)

plt.show()

## Montajes visuales

In [ ]:
def resize_for_montage(img, target_width=360):
    h, w = img.shape[:2]
    scale = target_width / float(w)
    new_size = (target_width, max(1, int(round(h * scale))))
    return cv2.resize(img, new_size, interpolation=cv2.INTER_AREA)


def add_label(img, label):
    out = img.copy()
    cv2.rectangle(out, (0, 0), (out.shape[1], 28), (0, 0, 0), -1)
    cv2.putText(out, label, (8, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)
    return out


def make_grid(images, labels, cols=3, target_width=360):
    tiles = [add_label(resize_for_montage(img, target_width), label) for img, label in zip(images, labels)]
    tile_h = max(tile.shape[0] for tile in tiles)
    tile_w = max(tile.shape[1] for tile in tiles)
    padded = []
    for tile in tiles:
        canvas = np.zeros((tile_h, tile_w, 3), dtype=np.uint8)
        canvas[:tile.shape[0], :tile.shape[1]] = tile
        padded.append(canvas)

    rows = []
    for start in range(0, len(padded), cols):
        row_tiles = padded[start:start + cols]
        while len(row_tiles) < cols:
            row_tiles.append(np.zeros((tile_h, tile_w, 3), dtype=np.uint8))
        rows.append(np.concatenate(row_tiles, axis=1))
    return np.concatenate(rows, axis=0)


labels = [
    f"{row['subject_id']} | angle {row['best_angle_deg']:.1f} | Dice {row['dice_rigid']:.3f}"
    for row in all_metrics
]
overlay_grid = make_grid([overlay_images[sid] for sid in SUBJECTS], labels)
contour_grid = make_grid([contour_images[sid] for sid in SUBJECTS], labels)

overlay_grid_path = OUTPUT_ROOT / 'rigid_mask_overlay_grid.png'
contour_grid_path = OUTPUT_ROOT / 'rigid_mask_contour_grid.png'
Image.fromarray(overlay_grid).save(overlay_grid_path)
Image.fromarray(contour_grid).save(contour_grid_path)

print('Montaje overlays:', overlay_grid_path)
print('Montaje contornos:', contour_grid_path)

plt.figure(figsize=(10, 6))
plt.imshow(contour_grid)
plt.axis('off')
plt.show()

## Lectura rapida

- Si `dice_rigid` sube frente a `dice_initial_angle_0`, la rotacion rigida aporta mejora.
- Si algun sujeto queda con Dice bajo o contornos claramente mal orientados, hay que cambiar la metrica de rotacion para ese caso.
- Los desajustes restantes, si son locales, son candidatos para afin suave o deformable.

In [ ]:
# Figura para la memoria: procedimiento de registro rigido basado en mascaras sobre SB013
import csv
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from PIL import Image


subject_id = 'SB013'
cwd = Path.cwd().resolve()
workspace = next(
    candidate
    for candidate in (cwd, *cwd.parents)
    if (candidate / 'Registration' / 'DeepLearning' / 'Imagenes_a_escala').exists()
)

input_dir = workspace / 'Registration' / 'DeepLearning' / 'Imagenes_a_escala' / subject_id
rigid_dir = (
    workspace
    / 'Registration'
    / 'DeepLearning'
    / 'Tecnicas_registration'
    / '00_baseline_clasico'
    / 'outputs'
    / 'outputs_registro_rigido_mascaras'
    / subject_id
)
figure_path = workspace / 'Fotos_Memoria' / 'SB013_proceso_registro_rigido.png'


def load_mask(path):
    return np.asarray(Image.open(path).convert('L')) > 0


def centroid_xy(mask):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        raise ValueError('La máscara está vacía')
    return np.array([xs.mean(), ys.mean()], dtype=np.float64)


def rigid_matrix_moving_to_fixed(moving_mask, fixed_mask, angle_deg):
    moving_centroid = centroid_xy(moving_mask)
    fixed_centroid = centroid_xy(fixed_mask)
    matrix = cv2.getRotationMatrix2D(tuple(moving_centroid), float(angle_deg), 1.0)
    matrix[:, 2] += fixed_centroid - moving_centroid
    return matrix


def warp_mask(mask, fixed_shape, matrix):
    warped = cv2.warpAffine(
        mask.astype(np.uint8) * 255,
        matrix,
        dsize=(fixed_shape[1], fixed_shape[0]),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    return warped > 0


def contour_mask(mask, thickness=2):
    mask_u8 = np.asarray(mask, dtype=np.uint8) * 255
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    output = np.zeros_like(mask_u8)
    cv2.drawContours(output, contours, -1, 255, int(thickness))
    return output > 0


def contour_overlay(fixed_mask, moving_mask):
    output = np.zeros((*fixed_mask.shape, 3), dtype=np.uint8)
    output[contour_mask(fixed_mask)] = np.array([255, 45, 45], dtype=np.uint8)
    output[contour_mask(moving_mask)] = np.array([0, 225, 235], dtype=np.uint8)
    return output


he_mask = load_mask(input_dir / f'{subject_id}_he_mask.png')
hsi_mask = load_mask(input_dir / f'{subject_id}_hsi_mask.png')
metrics = json.loads((rigid_dir / f'{subject_id}_rigid_metrics.json').read_text(encoding='utf-8'))
best_angle = float(metrics['best_angle_deg'])

initial_matrix = rigid_matrix_moving_to_fixed(he_mask, hsi_mask, angle_deg=0.0)
final_matrix = rigid_matrix_moving_to_fixed(he_mask, hsi_mask, angle_deg=best_angle)
initial_mask = warp_mask(he_mask, hsi_mask.shape, initial_matrix)
final_mask = warp_mask(he_mask, hsi_mask.shape, final_matrix)
initial_overlay = contour_overlay(hsi_mask, initial_mask)
final_overlay = contour_overlay(hsi_mask, final_mask)

with (rigid_dir / f'{subject_id}_angle_search.csv').open(encoding='utf-8', newline='') as handle:
    rows = list(csv.DictReader(handle))
angles = np.array([float(row['angle_deg']) for row in rows])
dices = np.array([float(row['dice']) for row in rows])
coarse_count = 181  # [-180, 180] con paso de 2 grados.
coarse_angles, coarse_dices = angles[:coarse_count], dices[:coarse_count]
fine_angles, fine_dices = angles[coarse_count:], dices[coarse_count:]

fig, axes = plt.subplots(1, 3, figsize=(15, 5.6))

axes[0].imshow(initial_overlay)
axes[0].set_title('(a) Alineación inicial de centroides')
axes[0].axis('off')

axes[1].plot(coarse_angles, coarse_dices, color='#3b4652', linewidth=1.4)
axes[1].scatter(fine_angles, fine_dices, color='#d97706', s=18, zorder=3)
axes[1].axvline(best_angle, color='#b91c1c', linestyle='--', linewidth=1.4)
axes[1].set_xlabel('Ángulo H&E → HSI (grados)')
axes[1].set_ylabel('Dice entre máscaras')
axes[1].set_title('(b) Búsqueda angular en dos etapas')
axes[1].set_xlim(-180, 180)
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.22)

zoom = inset_axes(axes[1], width='42%', height='38%', loc='lower left', borderpad=2.2)
zoom.plot(fine_angles, fine_dices, color='#d97706', marker='o', markersize=2.6, linewidth=1)
zoom.axvline(best_angle, color='#b91c1c', linestyle='--', linewidth=1)
zoom.set_xlim(best_angle - 3.2, best_angle + 3.2)
fine_min = max(0.0, float(fine_dices.min()) - 0.01)
fine_max = min(1.0, float(fine_dices.max()) + 0.005)
zoom.set_ylim(fine_min, fine_max)
zoom.tick_params(labelsize=7)
zoom.grid(alpha=0.2)

axes[2].imshow(final_overlay)
axes[2].set_title('(c) Transformación rígida seleccionada')
axes[2].axis('off')

legend = [
    Line2D([0], [0], color='#ff2d2d', lw=3, label='HSI fija'),
    Line2D([0], [0], color='#00e1eb', lw=3, label='H&E móvil'),
]
fig.legend(handles=legend, loc='upper center', ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.91))
fig.suptitle('Ejemplo del procedimiento de registro rígido basado en máscaras: SB013', fontsize=15, y=0.98)
fig.subplots_adjust(left=0.035, right=0.985, bottom=0.12, top=0.80, wspace=0.20)

figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close(fig)

print('Figura guardada en:', figure_path)
print('Caso ilustrativo:', subject_id)
print('Ángulo seleccionado:', best_angle, 'grados')
